## DRB Deep Investigation - Part 2

A one-query smoke test against an asynchronous deep-investigation endpoint, then the query run, then RACE grading with the Deep Research Bench scripts. Answers are appended to `drb_v4_deep_results.jsonl` (JSONL, one `{"id", "prompt", "article"}` object per line).

Bring your own endpoint and data: set `BENCHMARK_API_KEY` and replace `<MODAL_WORKSPACE>` and `<MODAL_APP_NAME>` in the endpoint URLs with your own deployment. To grade answers you already have, put them at `drb_v4_deep_results.jsonl` in the working directory. No data ships with this repository. Research submits refuse today (`RESEARCH_COST_BOUNDED = False`), and so does grading (`GRADER_COST_BOUNDED = False`).

In [ ]:
# --- Spend guard -----------------------------------------------------------
# Each submit starts a paid research job on Modal. JOB_WORST_CASE_USD (default 2.00, override with the environment
# variable of the same name) is the most one job is counted at, in USD. Each submit attempt (retries and failed
# submits included) is reserved at JOB_WORST_CASE_USD before it is sent and settled to it after.
# A submit that would take the run past RUN_BUDGET_USD (default 25.00, override with the environment variable of the
# same name) is refused, and the run stops. Status polls are not paid.
# One ledger root for every notebook: the directory in SPEND_LEDGER_DIR, default <repo root>/spend_ledger. One
# exclusive runner.lock in that root, so only one paid run exists at a time across all notebooks.
import atexit, json, math, os, subprocess, threading, uuid
from datetime import datetime, timezone

# --- Cost bound ----------------------------------------------------------------
# Spend policy: research-job submits stay off until the hosted endpoint has a proven per-job cost ceiling
# and a working way to cancel a running job. RESEARCH_COST_BOUNDED may be set to True only after both are shown. A
# polling timeout is not cancellation: the job keeps running and billing after this notebook stops waiting.
RESEARCH_COST_BOUNDED = False


def require_bounded_research():
    if RESEARCH_COST_BOUNDED is not True:
        raise RuntimeError(
            "Refusing to submit a research job: the hosted per-job cost ceiling and job cancellation are not proven. "
            "A polling timeout is not cancellation, so a job can keep running and billing after this notebook stops "
            "waiting, and the flat per-submit reservation is not an upper bound. Submits stay off until the operator shows "
            "a per-job ceiling and a working cancel. Nothing was sent.")




# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD and JOB_WORST_CASE_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_positive(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


RUN_BUDGET_USD = _env_positive("RUN_BUDGET_USD", 25.00)
RUN_ID = None                # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
JOB_WORST_CASE_USD = _env_positive("JOB_WORST_CASE_USD", 2.00)    # the most one submitted job is counted at, in USD


class BudgetExceeded(RuntimeError):
    pass


# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
LEDGER_ROOT = _ledger_root()
os.makedirs(LEDGER_ROOT, exist_ok=True)

# One runner at a time across all notebooks: one exclusive lock file in the shared ledger root. A second runner
# refuses to start while it exists. It is removed on a clean exit only. A stale lock is never cleared automatically.
RUNNER_LOCK = os.path.join(LEDGER_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be using the ledger, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)
RUN_DIR = os.path.join(LEDGER_ROOT, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)

_ledger_lock = threading.Lock()
_ledger = {}                 # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False       # set by a refused submit; no later submit is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)


def ledger_total():          # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def reserve_job(purpose):
    """Reserve one submit at JOB_WORST_CASE_USD before it is sent. Raises BudgetExceeded if no worst case is set,
    if the run is halted, or if the reservation would take the run past RUN_BUDGET_USD."""
    global _budget_halted
    if JOB_WORST_CASE_USD is None or not math.isfinite(JOB_WORST_CASE_USD) or JOB_WORST_CASE_USD <= 0:
        raise BudgetExceeded("not sent: JOB_WORST_CASE_USD is not a positive amount. The operator must set it first.")
    wc = round(float(JOB_WORST_CASE_USD), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so the {purpose} submit is refused")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "purpose": purpose, "worst_case": wc,
                 "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {purpose} submit, worst case ${wc:.4f}, would take the run past "
                         f"RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def settle_job(aid):
    """Settle one submit at its worst case. A submit is charged whether or not the job is accepted."""
    with _ledger_lock:
        e = _ledger[aid]
        e.update(state="settled", cost=e["worst_case"], cost_source="worst_case")
        _ledger_write({"id": aid, "state": "settled", "cost": e["worst_case"], "cost_source": "worst_case"})


print(f"Spend ledger: run {RUN_ID}, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}, charged so far ${ledger_total():.4f}")


# Quick smoke test — one query
require_bounded_research()
import requests, os, time
API_KEY = os.environ["BENCHMARK_API_KEY"]
aid = reserve_job("smoke")
try:
    resp = requests.post("https://<MODAL_WORKSPACE>--<MODAL_APP_NAME>-deep-investigation.modal.run", json={"api_key": API_KEY, "query": "What are FDA Class I recall criteria?"}, timeout=60)
finally:
    settle_job(aid)
data = resp.json()
print(data)
job_id = data["job_id"]
# Poll once
time.sleep(30)
status = requests.get("https://<MODAL_WORKSPACE>--<MODAL_APP_NAME>-status.modal.run", params={"job_id": job_id, "api_key": API_KEY}, timeout=30).json()
print(status)

In [ ]:
require_bounded_research()
import json, os, time, requests, subprocess
from pathlib import Path

API_KEY = os.environ["BENCHMARK_API_KEY"]
SUBMIT_URL = "https://<MODAL_WORKSPACE>--<MODAL_APP_NAME>-deep-investigation.modal.run"
STATUS_URL  = "https://<MODAL_WORKSPACE>--<MODAL_APP_NAME>-status.modal.run"
OUTPUT_FILE = "drb_v4_deep_results.jsonl"
QUERY_FILE  = "deep_research_bench/data/prompt_data/query.jsonl"

print("✓ BENCHMARK_API_KEY set")

_DRB_PIN = "f2735b5c3636759c22f9e936c0232de8bf545b67"   # same commit as GRADER_REVISION in the DRB grading notebooks
if not Path("deep_research_bench").exists():
    subprocess.run(["git", "clone", "https://github.com/Ayanami0730/deep_research_bench.git"], check=True)
    subprocess.run(["git", "-C", "deep_research_bench", "checkout", "-q", "--detach", _DRB_PIN], check=True)
# Read-only check on every run, fresh clone or not. An existing checkout is never moved, reset or changed here.
_drb_head = subprocess.run(["git", "-C", "deep_research_bench", "rev-parse", "HEAD"], capture_output=True,
                           text=True, check=True).stdout.strip()
if _drb_head != _DRB_PIN:
    raise RuntimeError(f"Refusing to read the DRB queries: deep_research_bench is at {_drb_head!r}, not the pinned "
                       f"commit {_DRB_PIN}. Nothing was changed. To use the pinned commit, run by hand:\n"
                       f"  git -C deep_research_bench fetch origin {_DRB_PIN}\n"
                       f"  git -C deep_research_bench checkout --detach {_DRB_PIN}")

queries = [json.loads(l) for l in open(QUERY_FILE) if l.strip()]
print(f"Loaded {len(queries)} queries.")

completed = set()
if Path(OUTPUT_FILE).exists():
    for l in open(OUTPUT_FILE):
        try: completed.add(json.loads(l.strip())["id"])
        except: pass
    if completed: print(f"Resuming: {len(completed)} done.")

remaining = [q for q in queries if q["id"] not in completed]
print(f"{len(remaining)} remaining, ~{len(remaining)*15/60:.1f} hrs, ~${len(remaining)*JOB_WORST_CASE_USD:.2f} worst case before retries (JOB_WORST_CASE_USD per submit attempt)")

def poll_job(job_id, max_wait=3600, interval=30):
    start = time.time()
    while time.time() - start < max_wait:
        try:
            resp = requests.get(
                STATUS_URL,
                params={"job_id": job_id, "api_key": API_KEY},
                timeout=30
            )
            data = resp.json()
            status = data.get("status", "")
            elapsed = time.time() - start
            print(f"  [{elapsed:.0f}s] status: {status} | {data.get('status_message', '')}")
            if status == "complete":
                return data["results"]["report"]
            elif status in ("failed", "error"):
                print(f"  Job failed: {data.get('status_message', 'unknown')}")
                return None
        except Exception as e:
            print(f"  Poll error: {e}")
        time.sleep(interval)
    print(f"  Timed out after {max_wait}s")
    return None

total_time, successes, failures = 0, 0, 0

for i, query in enumerate(remaining):
    qid   = query["id"]
    prompt = query.get("prompt", query.get("query", ""))
    print(f"\n[{i+1}/{len(remaining)}] {qid}: {prompt[:80]}...")
    article, elapsed = None, 0

    require_bounded_research()
    for attempt in range(1, 4):
        try:
            start = time.time()
            aid = reserve_job("submit")
            try:
                resp = requests.post(
                    SUBMIT_URL,
                    json={"api_key": API_KEY, "query": prompt},
                    timeout=60
                )
            finally:
                settle_job(aid)
            data = resp.json()
            if resp.status_code != 200 or data.get("error"):
                print(f"  Submit error (attempt {attempt}): {data.get('error', resp.status_code)}")
                time.sleep(60); continue
            job_id = data.get("job_id")
            if not job_id:
                print(f"  No job_id (attempt {attempt})")
                time.sleep(60); continue
            print(f"  ✓ Queued {job_id}, est. {data.get('estimated_minutes')} min — polling...")
            article = poll_job(job_id)
            elapsed = time.time() - start
            if article:
                print(f"  ✓ Complete: {len(article):,} chars, {elapsed:.0f}s")
                break
            time.sleep(60)
        except BudgetExceeded:
            raise
        except Exception as e:
            print(f"  {type(e).__name__}: {str(e)[:100]} (attempt {attempt})")
            time.sleep(60)

    if article:
        with open(OUTPUT_FILE, "a") as f:
            f.write(json.dumps({"id": qid, "prompt": prompt, "article": article}, ensure_ascii=False) + "\n")
        successes += 1; total_time += elapsed
    else:
        print(f"  ✗ FAILED: {qid}"); failures += 1

    if (i+1) % 10 == 0:
        avg = total_time / max(successes, 1)
        print(f"\n  --- {successes} done, {failures} failed, avg {avg:.0f}s/q, ETA ~{avg*(len(remaining)-i-1)/60:.0f} min ---\n")

print(f"\n{'='*50}")
print(f"  DONE: {successes} ok, {failures} failed, {total_time/60:.1f} min total")
print(f"{'='*50}")

In [ ]:
# Bring your own data: the cells below read drb_v4_deep_results.jsonl in the working directory
# (JSONL, one {"id", "prompt", "article"} object per line). The run cells write it, or put your
# own results file there before running the cells below.


In [ ]:
require_bounded_research()
import json, os, time, requests, subprocess
from pathlib import Path

API_KEY = os.environ["BENCHMARK_API_KEY"]
SUBMIT_URL = "https://<MODAL_WORKSPACE>--<MODAL_APP_NAME>-deep-investigation.modal.run"
STATUS_URL  = "https://<MODAL_WORKSPACE>--<MODAL_APP_NAME>-status.modal.run"
OUTPUT_FILE = "drb_v4_deep_results.jsonl"
QUERY_FILE  = "deep_research_bench/data/prompt_data/query.jsonl"

print("✓ BENCHMARK_API_KEY set")

_DRB_PIN = "f2735b5c3636759c22f9e936c0232de8bf545b67"   # same commit as GRADER_REVISION in the DRB grading notebooks
if not Path("deep_research_bench").exists():
    subprocess.run(["git", "clone", "https://github.com/Ayanami0730/deep_research_bench.git"], check=True)
    subprocess.run(["git", "-C", "deep_research_bench", "checkout", "-q", "--detach", _DRB_PIN], check=True)
# Read-only check on every run, fresh clone or not. An existing checkout is never moved, reset or changed here.
_drb_head = subprocess.run(["git", "-C", "deep_research_bench", "rev-parse", "HEAD"], capture_output=True,
                           text=True, check=True).stdout.strip()
if _drb_head != _DRB_PIN:
    raise RuntimeError(f"Refusing to read the DRB queries: deep_research_bench is at {_drb_head!r}, not the pinned "
                       f"commit {_DRB_PIN}. Nothing was changed. To use the pinned commit, run by hand:\n"
                       f"  git -C deep_research_bench fetch origin {_DRB_PIN}\n"
                       f"  git -C deep_research_bench checkout --detach {_DRB_PIN}")

queries = [json.loads(l) for l in open(QUERY_FILE) if l.strip()]
print(f"Loaded {len(queries)} queries.")

# Match on prompt text — bulletproof regardless of ID format
completed_prompts = set()
if Path(OUTPUT_FILE).exists():
    for l in open(OUTPUT_FILE):
        try:
            entry = json.loads(l.strip())
            completed_prompts.add(entry.get("prompt","").strip()[:100])
        except: pass
    print(f"Resuming: {len(completed_prompts)} done.")

remaining = []
for q in queries:
    prompt = q.get("prompt", q.get("query", "")).strip()
    if prompt[:100] not in completed_prompts:
        remaining.append(q)

print(f"{len(remaining)} remaining, ~{len(remaining)*13/60:.1f} hrs")

def poll_job(job_id, max_wait=3600, interval=30):
    start = time.time()
    while time.time() - start < max_wait:
        try:
            resp = requests.get(STATUS_URL, params={"job_id": job_id, "api_key": API_KEY}, timeout=30)
            data = resp.json()
            status = data.get("status", "")
            print(f"  [{time.time()-start:.0f}s] status: {status} | {data.get('status_message','')}")
            if status == "complete":
                return data["results"]["report"]
            elif status in ("failed", "error"):
                print(f"  Job failed: {data.get('status_message','unknown')}")
                return None
        except Exception as e:
            print(f"  Poll error: {e}")
        time.sleep(interval)
    print(f"  Timed out after {max_wait}s")
    return None

total_time, successes, failures = 0, 0, 0

for i, query in enumerate(remaining):
    qid = query.get("id", i)
    prompt = query.get("prompt", query.get("query", "")).strip()
    print(f"\n[{i+1}/{len(remaining)}] {qid}: {prompt[:80]}...")
    article, elapsed = None, 0

    require_bounded_research()
    for attempt in range(1, 4):
        try:
            start = time.time()
            aid = reserve_job("submit")
            try:
                resp = requests.post(SUBMIT_URL, json={"api_key": API_KEY, "query": prompt}, timeout=60)
            finally:
                settle_job(aid)
            data = resp.json()
            if resp.status_code != 200 or data.get("error"):
                print(f"  Submit error (attempt {attempt}): {data.get('error', resp.status_code)}")
                time.sleep(60); continue
            job_id = data.get("job_id")
            if not job_id:
                print(f"  No job_id (attempt {attempt})")
                time.sleep(60); continue
            print(f"  ✓ Queued {job_id}, est. {data.get('estimated_minutes')} min — polling...")
            article = poll_job(job_id)
            elapsed = time.time() - start
            if article:
                print(f"  ✓ Complete: {len(article):,} chars, {elapsed:.0f}s")
                break
            time.sleep(60)
        except BudgetExceeded:
            raise
        except Exception as e:
            print(f"  {type(e).__name__}: {str(e)[:100]} (attempt {attempt})")
            time.sleep(60)

    if article:
        with open(OUTPUT_FILE, "a") as f:
            f.write(json.dumps({"id": qid, "prompt": prompt, "article": article}, ensure_ascii=False) + "\n")
        successes += 1; total_time += elapsed
    else:
        print(f"  ✗ FAILED: {qid}"); failures += 1

    if (i+1) % 10 == 0:
        avg = total_time / max(successes, 1)
        print(f"\n  --- {successes} done, {failures} failed, avg {avg:.0f}s/q, ETA ~{avg*(len(remaining)-i-1)/60:.0f} min ---\n")

print(f"\n{'='*50}")
print(f"  DONE: {successes} ok, {failures} failed, {total_time/60:.1f} min total")
print(f"{'='*50}")

In [ ]:
import os, re, subprocess, shutil, pathlib

# --- Grader revision ----------------------------------------------------------
# GRADER_REVISION is the grader checkout the results are graded with: the full 40-character commit SHA of
# Ayanami0730/deep_research_bench. This is the current upstream commit at the time of the pin (2026-11-05).
# Every grading step refuses to run while this is None or is
# not a full lowercase SHA, or while the checkout is not exactly at it. The notebook never moves the checkout itself.
# The pinned grader is the GPT-era version (OpenAI or OpenRouter models). Scores from it are not comparable with
# scores from the earlier Gemini-based grader.
GRADER_REVISION = "f2735b5c3636759c22f9e936c0232de8bf545b67"
GRADER_REPO = "https://github.com/Ayanami0730/deep_research_bench.git"
GRADER_DIR = "deep_research_bench"

# --- Cost bound ----------------------------------------------------------------
# Spend policy: grading stays off until every paid call the grader makes can be metered or bounded before
# it is dispatched. GRADER_COST_BOUNDED may be set to True only after that is true for every internal paid grader call
# (RACE scoring, cleaning, retries, FACT steps). Until then every grading entry in this notebook refuses.
GRADER_COST_BOUNDED = False


def require_bounded_grading():
    if GRADER_COST_BOUNDED is not True:
        raise RuntimeError(
            "Refusing to grade: the grader makes its own model calls in a subprocess (RACE scoring, cleaning, retries "
            "and the FACT steps), and this notebook cannot meter or cap any of them before they are sent. The flat "
            "per-step reservation is not an upper bound: a single call with a 64000-token output lists at about 7.20 USD "
            "on openrouter before any input cost, and scoring and cleaning retry. Grading stays off until those calls "
            "are bounded. Nothing was installed, copied or sent.")


def require_grader_revision():
    if GRADER_REVISION is None or not re.fullmatch(r"[0-9a-f]{40}", str(GRADER_REVISION)):
        raise RuntimeError("GRADER_REVISION is not a full 40-character commit SHA. Refusing to grade. "
                           "The operator must set it first.")


def verify_grader_tree_clean():
    """Read-only working-tree integrity check for the grader checkout. The checkout verification calls it,
    so every grader entry point and every grader subprocess is covered. It refuses when a tracked file is modified,
    deleted, staged or has a changed mode, when any untracked file (ignored files included) is not a plain data file,
    or when the checkout holds a nested .git or a symlink. Allowed untracked files: .jsonl or .json under
    data/test_data/raw_data/ and data/test_data/cleaned_data/, and .jsonl, .json, .txt or .log under results/, none with an executable bit. Nothing is
    cleaned, fetched or checked out here, and git is only asked to read."""
    import stat
    fix = (f"What to do by hand: inspect each item listed above, then run  git -C {GRADER_DIR} status  and look at the "
           f"tree. If you did not make the change yourself, re-clone {GRADER_DIR}, or run  git -C {GRADER_DIR} checkout "
           f"--detach {GRADER_REVISION}  yourself (that drops tracked edits), and remove stray untracked files yourself. "
           "This notebook never cleans or checks out anything itself.")
    keep = ("PATH", "HOME", "USERPROFILE", "XDG_CONFIG_HOME", "SYSTEMROOT", "TMPDIR")
    env = {k: os.environ[k] for k in keep if k in os.environ}
    env.update(LC_ALL="C", GIT_TERMINAL_PROMPT="0")
    base = ["git", "-C", GRADER_DIR, "--no-optional-locks", "-c", "core.fsmonitor=false", "-c", "core.fileMode=true"]

    def git(*args):
        try:
            out = subprocess.run(base + list(args), capture_output=True, check=True, env=env, timeout=120).stdout
        except (OSError, subprocess.SubprocessError) as e:
            raise RuntimeError(f"Refusing to grade: cannot read the working tree state of {GRADER_DIR} "
                               f"({type(e).__name__}). {fix}")
        return out.encode() if isinstance(out, str) else out

    def entries(raw):
        return [x.decode("utf-8", "surrogateescape") for x in raw.split(b"\0") if x]

    problems = []
    for e in entries(git("status", "--porcelain=v1", "-z", "--untracked-files=no", "--ignore-submodules=none")):
        problems.append(f"tracked file changed (modified, deleted, staged or mode changed): {e}")
    for e in entries(git("ls-files", "-v", "-z")):
        if not e.startswith("H "):
            problems.append(f"tracked file hidden from status (assume-unchanged, skip-worktree or unmerged): {e[2:]}")
    for p in entries(git("ls-files", "-z", "--others")):
        parts = p.split("/")
        ext = os.path.splitext(parts[-1])[1]
        if "" in parts or ".." in parts or ".git" in parts:
            problems.append(f"nested repository or unsafe path: {p}")
            continue
        ok = ((p.startswith("data/test_data/raw_data/") and ext in (".jsonl", ".json")) or
              (p.startswith("data/test_data/cleaned_data/") and ext in (".jsonl", ".json")) or
              (p.startswith("results/") and ext in (".jsonl", ".json", ".txt", ".log")))
        if not ok:
            problems.append(f"untracked file that is not an allowed data file: {p}")
            continue
        try:
            st = os.lstat(os.path.join(GRADER_DIR, *parts))
        except OSError:
            problems.append(f"untracked file that cannot be inspected: {p}")
            continue
        if stat.S_ISLNK(st.st_mode):
            problems.append(f"untracked symlink: {p}")
        elif not stat.S_ISREG(st.st_mode):
            problems.append(f"untracked entry that is not a regular file: {p}")
        elif st.st_mode & 0o111:
            problems.append(f"untracked data file with an executable bit: {p}")
    for root, dirs, files in os.walk(GRADER_DIR):
        if root == GRADER_DIR and ".git" in dirs:
            dirs.remove(".git")
        for n in list(dirs) + list(files):
            if n == ".git":
                problems.append(f"nested .git: {os.path.join(root, n)}")
        if ".git" in dirs:
            dirs.remove(".git")
    if problems:
        shown = "\n".join("  - " + x for x in problems[:20]) + (f"\n  ... and {len(problems) - 20} more" if len(problems) > 20 else "")
        raise RuntimeError(f"Refusing to grade: the grader checkout in {GRADER_DIR} differs from the pinned commit in a "
                           f"way that could run different code with this process's environment. Found:\n{shown}\n{fix}")


def verify_grader_checkout():
    """Read-only check that the grader checkout is at GRADER_REVISION, that its working tree is clean (see
    verify_grader_tree_clean), and that its origin is GRADER_REPO.
    Nothing is cloned, fetched or checked out here. On any mismatch it refuses and prints the commands to run by hand."""
    require_grader_revision()
    fix = (f"  git clone {GRADER_REPO} {GRADER_DIR}      (only if {GRADER_DIR} does not exist)\n"
           f"  git -C {GRADER_DIR} remote set-url origin {GRADER_REPO}\n"
           f"  git -C {GRADER_DIR} fetch origin {GRADER_REVISION}\n"
           f"  git -C {GRADER_DIR} checkout --detach {GRADER_REVISION}")
    try:
        head = subprocess.run(["git", "-C", GRADER_DIR, "rev-parse", "HEAD"], capture_output=True, text=True,
                              check=True).stdout.strip()
        origin = subprocess.run(["git", "-C", GRADER_DIR, "remote", "get-url", "origin"], capture_output=True,
                                text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError) as e:
        raise RuntimeError(f"Refusing to grade: cannot read the grader checkout in {GRADER_DIR} ({e}). "
                           f"The operator must run by hand:\n{fix}")
    if head != GRADER_REVISION or origin != GRADER_REPO:
        raise RuntimeError(f"Refusing to grade: the grader checkout is at {head} with origin {origin}, but it must be "
                           f"at {GRADER_REVISION} with origin {GRADER_REPO}. The operator must run by hand:\n{fix}")
    verify_grader_tree_clean()


require_bounded_grading()
verify_grader_checkout()

# Confirmed listed prices in USD per million tokens, (input, output), the highest that can apply, retrieved 2026-11-05.
# Keys are route|model. Sources: https://openrouter.ai/api/v1/models/openai/gpt-5.5/endpoints and
# https://openrouter.ai/api/v1/models/openai/gpt-5.6-luna/endpoints (openrouter_http);
# https://developers.openai.com/api/docs/models/gpt-5.5 and https://developers.openai.com/api/docs/pricing (openai_direct).
# This notebook runs RACE scoring only, so the fact-checking model is not needed here and is not listed.
PRICE = {
    "openrouter_http|openai/gpt-5.5": (25.0, 112.5),
    "openrouter_http|openai/gpt-5.6-luna": (0.8, 3.6),
    "openai_direct|gpt-5.5": (10.0, 45.0),
    "openai_direct|gpt-5.6-luna": (0.4, 1.8),
}
# Output cap per grader call, sent to the grader as MAX_OUTPUT_TOKENS (the grader's own default is also 64000).
GRADER_MAX_OUTPUT_TOKENS = 64000
# At the pinned commit deepresearch_bench_race.py calls two models: RACE_MODEL (scoring) and CLEAN_MODEL (cleaning).
# Per backend: (route, name of the key variable utils/api.py reads, default RACE_MODEL, default CLEAN_MODEL).
_GRADER_BACKENDS = {"openrouter": ("openrouter_http", "OPENROUTER_API_KEY", "openai/gpt-5.5", "openai/gpt-5.6-luna"),
                    "openai": ("openai_direct", "OPENAI_API_KEY", "gpt-5.5", "gpt-5.6-luna")}


def require_grader_prices():
    """Refuse before any paid call unless the backend is known, its key variable is set, no base URL override is set,
    and every model the grader will call has a confirmed price in PRICE. Only variable NAMES are checked or printed,
    never values. Returns the environment to pass to the grader so it calls exactly the models that were checked."""
    backend = os.environ.get("LLM_BACKEND", "openrouter").lower()
    if backend not in _GRADER_BACKENDS:
        raise RuntimeError(f"Refusing to grade: LLM_BACKEND={backend!r} is not one of {sorted(_GRADER_BACKENDS)}, "
                           "so no price can be confirmed.")
    route, key_var, d_race, d_clean = _GRADER_BACKENDS[backend]
    if not os.environ.get(key_var, "").strip():
        raise RuntimeError(f"Refusing to grade: {key_var} is not set or is blank. The grader reads exactly this "
                           f"variable when LLM_BACKEND={backend}. Set it in the environment (value not shown).")
    for url_var in ("OPENROUTER_BASE_URL", "OPENAI_BASE_URL"):
        if url_var in os.environ:
            raise RuntimeError(f"Refusing to grade: {url_var} is set. The prices in PRICE are for the official host, "
                               "and an override would also redirect the API key. Unset it.")
    models = {"RACE_MODEL": os.environ.get("RACE_MODEL", d_race), "CLEAN_MODEL": os.environ.get("CLEAN_MODEL", d_clean)}
    for var, m in models.items():
        if f"{route}|{m}" not in PRICE:
            raise RuntimeError(f"Refusing to grade: no confirmed price for {var}={m!r} on route {route}. Add "
                               f"'{route}|{m}' to PRICE in this notebook with its source URL and date, or set {var} "
                               "to a model that is listed in PRICE.")
    return dict(LLM_BACKEND=backend, MAX_OUTPUT_TOKENS=str(GRADER_MAX_OUTPUT_TOKENS), **models)


require_grader_prices()

# Install deps
subprocess.run(["pip", "install", "-r", "deep_research_bench/requirements.txt", "--quiet"], check=True)
print("✓ Dependencies installed")

# Copy results
src = pathlib.Path("drb_v4_deep_results.jsonl")
dst = pathlib.Path("deep_research_bench/data/test_data/raw_data/deep-investigation-v4.jsonl")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)
print(f"✓ Results copied: {dst.stat().st_size:,} bytes")

# Run grading — target_model is POSITIONAL not a flag
# --output_dir puts the results where this notebook reads them.
# The grader runs in a subprocess and cannot be metered per call, so the step is one flat JOB_WORST_CASE_USD
# reservation (default 2.00). That is not an upper bound: one gpt-5.5 call at 64000 output tokens on openrouter lists at
# about 7.2 USD and scoring and cleaning retry. The spend policy is to refuse until bounded (GRADER_COST_BOUNDED), so this step does not run today.
require_bounded_grading()
verify_grader_checkout()
_grader_env = dict(os.environ, **require_grader_prices(), PYTHONDONTWRITEBYTECODE="1")
with open("grading_log.txt", "w") as log:
    aid = reserve_job("grade")
    try:
        r = subprocess.run(
            ["python", "deepresearch_bench_race.py", "deep-investigation-v4", "--output_dir", "results/race/deep-investigation-v4"],
            cwd="deep_research_bench",
            stdout=log, stderr=subprocess.STDOUT,
            env=_grader_env,
            timeout=7200
        )
    finally:
        settle_job(aid)

print(f"Exit code: {r.returncode}")
print(open("grading_log.txt").read()[-3000:])

rpath = pathlib.Path("deep_research_bench/results/race/deep-investigation-v4/race_result.txt")
if rpath.exists():
    print("\n=== RACE RESULTS ===")
    print(open(rpath).read())
else:
    print("No race_result.txt yet")

"done"